# Day 1, Session 4 -- Exercises: Lab Review & Integration

**Engineering context:** You are building a consulting AI system that combines all Day 1 techniques -- API calls, prompt engineering, and evaluation -- into production-ready workflows.

**How to use this notebook:**
- Each exercise builds on patterns from the demo notebook (`D1S4_demos.ipynb`)
- Follow the numbered TODO steps in order
- Hints are provided in collapsible sections -- try first, then peek if stuck
- Exercises 1-2 are required; Optional exercises are stretch goals

**Structure:**
- Exercise 1 (Easy): Consulting Deliverable Generator
- Exercise 2 (Easy): Multi-Persona Comparison
- Optional Exercise 1 (Intermediate): Consulting AI Readiness Assessment
- Optional Exercise 2 (Intermediate): System Architecture Design

In [ ]:
!pip install -q openai pandas python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import openai
import json
import os
import time
import numpy as np
import pandas as pd

client = openai.OpenAI()

# Pricing dict (reused from demos)
PRICING = {
    'gpt-4o-mini': {'input': 0.15 / 1_000_000, 'output': 0.60 / 1_000_000},
    'gpt-4o': {'input': 2.50 / 1_000_000, 'output': 10.00 / 1_000_000},
}

# G-Eval scorer (reused from Demo 1)
def g_eval_quick(question, response, criterion, description):
    """Lightweight G-Eval scorer (from Session 3)."""
    prompt = f"""Rate this consulting response on {criterion} (1-5).
Criterion: {description}
Question: {question}
Response: {response}
Return JSON with 'score' (1-5) and 'reasoning' (one sentence)."""
    result = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0, max_tokens=200
    )
    return json.loads(result.choices[0].message.content)

print("All imports successful!")
print(f"Model: {os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')}")

## Recap from Demos

In the demos you saw how to:
- **Demo 1:** Build an end-to-end consulting pipeline (generate + auto-evaluate with G-Eval)
- **Demo 2:** Analyze cost-quality trade-offs across API configurations
- **Demo 3:** Compare raw API calls vs LangChain chains (Day 2 preview)

Now you will build these patterns yourself.

---
## Exercise 1 (Easy): Build a Consulting Deliverable Generator

**Reference:** Demo 1 (End-to-End Pipeline)

> **Hint:** The function has 3 steps: (1) select persona from dict, (2) generate with API call, (3) evaluate with `g_eval_quick` on 3 criteria. Most code is provided -- you just fill in the evaluation loop.

Build a function that takes a client question and practice area, generates a structured consulting response using a role-based persona, auto-evaluates quality, and returns a full deliverable package.

**Steps:**
1. Use the provided PERSONAS dict and API call (already done)
2. Fill in the evaluation loop that scores with `g_eval_quick`
3. Calculate average score and set verdict
4. Test with the provided questions

In [ ]:
# Exercise 1: Build a Consulting Deliverable Generator

# Persona definitions (provided)
PERSONAS = {
    "Strategy": "You are a McKinsey senior strategy consultant. Provide structured, MECE analysis with clear strategic recommendations backed by market data.",
    "Operations": "You are a McKinsey operations expert. Focus on process optimization, supply chain efficiency, and operational KPIs with quantitative targets.",
    "Digital": "You are a McKinsey digital transformation specialist. Emphasize technology adoption, digital capabilities, and data-driven decision making.",
    "M&A": "You are a McKinsey M&A advisor. Cover deal rationale, valuation frameworks, synergy estimation, and integration planning."
}

def generate_deliverable(question, practice='Strategy', review=True):
    """Generate a consulting deliverable with auto-evaluation."""
    persona = PERSONAS.get(practice, PERSONAS["Strategy"])
    
    # Step 1: Generate response (provided)
    start = time.time()
    response = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': persona},
            {'role': 'user', 'content': question}
        ],
        temperature=0, max_tokens=500
    )
    latency = round((time.time() - start) * 1000)
    content = response.choices[0].message.content
    usage = response.usage

    # Step 2: Estimate cost (provided)
    model_name = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
    cost = 0
    if model_name in PRICING:
        cost = usage.prompt_tokens * PRICING[model_name]['input'] + usage.completion_tokens * PRICING[model_name]['output']

    # Step 3: Auto-evaluate if review=True
    scores = {}
    avg_score = 0
    verdict = "NOT EVALUATED"
    
    if review:
        criteria = {
            'MECE Structure': 'Response breaks down the problem into MECE categories.',
            'Actionability': 'Recommendations are specific, prioritized, and implementable.',
            'Executive Readiness': 'Output is polished enough for a C-suite presentation.'
        }
        
        # TODO: Loop over criteria, call g_eval_quick for each, collect scores
        # Hint: for name, desc in criteria.items():
        #     result = g_eval_quick(question, content, name, desc)
        #     scores[name] = result.get('score', 0)
        
        # YOUR CODE HERE
        
        # TODO: Calculate avg_score and set verdict
        # Hint: avg_score = round(sum(scores.values()) / len(scores), 1) if scores else 0
        # Hint: verdict = 'PARTNER-READY' if avg_score >= 4.0 else 'NEEDS REVISION' if avg_score >= 3.0 else 'REWORK'
        
        # YOUR CODE HERE
        pass

    result = {
        'practice': practice, 'content': content, 'scores': scores,
        'avg_score': avg_score, 'verdict': verdict,
        'latency_ms': latency, 'tokens': usage.total_tokens,
        'cost': round(cost, 6)
    }
    
    print(f"[{practice}] {verdict} | Score: {avg_score}/5 | {latency}ms | {usage.total_tokens} tokens | ${cost:.6f}")
    return result

# Test (uncomment when ready):
# d1 = generate_deliverable('How should a luxury retailer enter the Middle East market?', 'Strategy')
# d2 = generate_deliverable('What are the Day-1 integration priorities for a healthcare acquisition?', 'M&A')

<details>
<summary><strong>Hint 1:</strong> Evaluation loop</summary>

```python
if review:
    criteria = {
        'MECE Structure': 'Response breaks down the problem into MECE categories.',
        'Actionability': 'Recommendations are specific, prioritized, and implementable.',
        'Executive Readiness': 'Output is polished enough for a C-suite presentation.'
    }
    for name, desc in criteria.items():
        result = g_eval_quick(question, content, name, desc)
        scores[name] = result.get('score', 0)
    
    avg_score = round(sum(scores.values()) / len(scores), 1)
    verdict = 'PARTNER-READY' if avg_score >= 4.0 else 'NEEDS REVISION' if avg_score >= 3.0 else 'REWORK'
```
</details>

### Expected Output

```
[Strategy] PARTNER-READY | Score: 4.3/5 | 8500ms | 549 tokens | $0.000305
[M&A] PARTNER-READY | Score: 4.7/5 | 9200ms | 655 tokens | $0.000369
```

Exact scores and latency will vary, but both should typically score 4+/5.

---
## Exercise 2 (Easy): Build a Multi-Persona Comparison

**Reference:** Demo 1 (uses `generate_deliverable` from Exercise 1)

> **Hint:** Call `generate_deliverable()` in a loop for each practice area. Build a DataFrame from results and sort by score to find the best perspective.

Build a function that generates responses from different practice-area personas for the same question, evaluates each, and recommends which perspective is strongest.

**Steps:**
1. Loop through practices calling `generate_deliverable()` (provided)
2. Build a DataFrame and sort by average score
3. Identify and print the recommended perspective

In [ ]:
# Exercise 2: Multi-Persona Comparison

def compare_perspectives(question, practices=None):
    """Compare responses from different McKinsey practice perspectives."""
    if practices is None:
        practices = ['Strategy', 'Operations', 'Digital', 'M&A']
    
    # Step 1: Generate deliverable for each practice (provided)
    results = []
    for practice in practices:
        d = generate_deliverable(question, practice, review=True)
        results.append({
            'Practice': practice,
            'Avg Score': d['avg_score'],
            'Verdict': d['verdict'],
            'Tokens': d['tokens'],
            'Cost': f"${d['cost']:.6f}"
        })
    
    # TODO Step 2: Build DataFrame and sort by 'Avg Score' descending
    # Hint: df = pd.DataFrame(results).sort_values('Avg Score', ascending=False)
    # Hint: print(df.to_string(index=False))
    
    # YOUR CODE HERE
    
    # TODO Step 3: Identify and print the best perspective
    # Hint: best = df.iloc[0]
    # Hint: print(f"\nRecommendation: Lead with {best['Practice']} perspective (score: {best['Avg Score']}/5)")
    
    # YOUR CODE HERE
    pass

# Test (uncomment when ready):
# comparison = compare_perspectives(
#     'How should a regional hospital network respond to declining inpatient volumes?'
# )

<details>
<summary><strong>Hint:</strong> Full solution</summary>

```python
df = pd.DataFrame(results).sort_values('Avg Score', ascending=False)
print(f"\n{'=' * 60}")
print("RANKING:")
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\nRecommendation: Lead with {best['Practice']} perspective (score: {best['Avg Score']}/5)")
return df
```
</details>

### Expected Output

```
[Strategy] PARTNER-READY | Score: 4.3/5 | ...
[Operations] PARTNER-READY | Score: 4.7/5 | ...
[Digital] PARTNER-READY | Score: 4.3/5 | ...
[M&A] PARTNER-READY | Score: 5.0/5 | ...

============================================================
RANKING:
  Practice  Avg Score       Verdict  ...
       M&A        5.0 PARTNER-READY  ...
Operations        4.7 PARTNER-READY  ...
  Strategy        4.3 PARTNER-READY  ...
   Digital        4.3 PARTNER-READY  ...

Recommendation: Lead with M&A perspective (score: 5.0/5)
```

---
---
# Optional Exercises

> **Hint:** Review the demos for the patterns to follow. The optional exercises combine multiple demo patterns.

These exercises are at an intermediate level. Attempt them if you finish the required exercises early.

---
## Optional Exercise 1 (Intermediate): Consulting AI Readiness Assessment

**Reference:** Demo 1 + Demo 2 (Pipeline + Cost analysis)

> **Hint:** Define a test suite dict of category -> question. For each, generate + score on 2 criteria. Average the scores per category. PASS if avg >= 3.5.

Build a function that runs a test suite of consulting questions across categories and produces a readiness scorecard with pass/fail per category and an overall verdict.

In [ ]:
# Optional Exercise 1: Consulting AI Readiness Assessment

# TODO: Build readiness_assessment() that:
#   1. Defines a test suite of 4 consulting questions across categories
#      (Market Sizing, Due Diligence, Cost Optimization, Strategy)
#   2. Generates a response for each question
#   3. Scores each with g_eval_quick on "Quality" and "Actionability"
#   4. Marks each category as PASS (avg >= 3.5) or FAIL
#   5. Prints a readiness scorecard with overall verdict
#
# Verdict logic:
#   - All pass + avg >= 4.0 -> "READY FOR DEPLOYMENT"
#   - >= 75% pass -> "CONDITIONAL"
#   - else -> "NOT READY"

def readiness_assessment(model_config=None):
    """Run a comprehensive readiness assessment for consulting AI deployment."""
    if model_config is None:
        model_config = {'model': os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'), 'temperature': 0}
    
    test_suite = {
        'Market Sizing': 'Estimate the total addressable market for autonomous trucking in North America by 2030.',
        'Due Diligence': 'What are the key risks in acquiring a mid-size European fintech company?',
        'Cost Optimization': 'Identify 3 cost reduction levers for a manufacturing company with declining margins.',
        'Strategy': 'How should a traditional retailer respond to direct-to-consumer competition?',
    }
    
    # YOUR CODE HERE
    pass

# Test (uncomment when ready):
# scores = readiness_assessment()

<details>
<summary><strong>Hint:</strong> Readiness assessment loop</summary>

```python
category_scores = {}
for category, question in test_suite.items():
    response = client.chat.completions.create(
        model=model_config['model'], temperature=model_config['temperature'],
        messages=[
            {'role': 'system', 'content': 'You are a McKinsey senior consultant. Be structured and data-driven.'},
            {'role': 'user', 'content': question}
        ],
        max_tokens=400
    )
    content = response.choices[0].message.content
    s1 = g_eval_quick(question, content, 'Quality', 'Structured, accurate, consulting-ready.')['score']
    s2 = g_eval_quick(question, content, 'Actionability', 'Specific and implementable.')['score']
    avg = (s1 + s2) / 2
    status = 'PASS' if avg >= 3.5 else 'FAIL'
    category_scores[category] = {'score': avg, 'status': status}
    print(f"  {category:20s} {avg:.1f}/5 [{status}]")

overall = np.mean([v['score'] for v in category_scores.values()])
passed = sum(1 for v in category_scores.values() if v['status'] == 'PASS')
if passed == len(category_scores) and overall >= 4.0:
    print("\nVERDICT: READY FOR DEPLOYMENT")
elif passed >= len(category_scores) * 0.75:
    print("\nVERDICT: CONDITIONAL")
else:
    print("\nVERDICT: NOT READY")
return category_scores
```
</details>

---
## Optional Exercise 2 (Intermediate): System Architecture Design

**Reference:** All 3 Demos

> **Hint:** Think of 4 components: Input Router, Analysis Generator, Quality Gate, Knowledge Base. For each, map which Day 1 technique it uses and what Day 2 framework would improve it.

Design an AI system architecture for a McKinsey consulting use case. Document it as a structured specification dict with components, model selection, and quality thresholds.

In [ ]:
# Optional Exercise 2: System Architecture Design

# TODO: Complete the system_design dict by:
#   1. Adding 'quality_gate' and 'knowledge_base' components
#   2. Adding model configs for quality_gate and knowledge_base
#   3. Print the complete design

system_design = {
    'name': 'McKinsey Engagement AI Assistant',
    'use_case': 'AI-assisted consulting deliverable generation with quality gates',
    'components': {
        'input_router': {
            'purpose': 'Classify incoming client requests by practice area',
            'technique': 'LLM classification (Session 1 API calls, T=0)',
            'day2_upgrade': 'LangChain intent classifier chain'
        },
        'analysis_generator': {
            'purpose': 'Generate structured consulting analysis',
            'technique': 'Role-based prompting with personas (Session 2)',
            'day2_upgrade': 'LangChain LCEL chain with reusable templates'
        },
        # TODO: Add 'quality_gate' component
        # Hint: purpose = "Auto-evaluate deliverable quality"
        # Hint: technique = "G-Eval scoring (Session 3)"
        # Hint: day2_upgrade = "LangGraph conditional edges for pass/fail routing"
        
        # TODO: Add 'knowledge_base' component
        # Hint: purpose = "Store and retrieve past engagement insights"
        # Hint: technique = "Embeddings + cosine similarity (Session 1)"
        # Hint: day2_upgrade = "RAG with ChromaDB (Day 3)"
    },
    'model_selection': {
        'router': {'model': 'gpt-4o-mini', 'temperature': 0, 'max_tokens': 50},
        'generator': {'model': 'gpt-4o-mini', 'temperature': 0, 'max_tokens': 500},
        # TODO: Add quality_gate and knowledge_base model configs
    },
    'quality_thresholds': {
        'partner_ready': 4.0,
        'needs_revision': 3.0,
    },
}

print(json.dumps(system_design, indent=2))

<details>
<summary><strong>Hint:</strong> Missing components</summary>

```python
'quality_gate': {
    'purpose': 'Auto-evaluate deliverable quality before client delivery',
    'technique': 'G-Eval scoring on MECE, actionability, executive readiness (Session 3)',
    'day2_upgrade': 'LangGraph conditional edges for pass/fail routing with self-correction loops'
},
'knowledge_base': {
    'purpose': 'Store and retrieve past engagement insights and industry data',
    'technique': 'Embeddings + cosine similarity search (Session 1)',
    'day2_upgrade': 'Full RAG pipeline with ChromaDB vector store and reranking (Day 3)'
}
```
</details>

---
## Summary

| Exercise | Difficulty | Key Skill |
|----------|-----------|-----------|
| 1 | Easy | End-to-end pipeline with persona selection and auto-evaluation |
| 2 | Easy | Multi-persona comparison and ranking with DataFrames |
| Opt 1 | Intermediate | Readiness assessment across consulting categories |
| Opt 2 | Intermediate | System architecture design mapping Day 1 to Day 2 |

**Key Takeaways:**
- Real consulting AI systems combine generation + evaluation in a single pipeline
- Auto-evaluation with G-Eval enables quality gates without human review
- Different practice areas benefit from different personas and configurations
- Cost estimation is critical for scoping AI-augmented engagements

**What's Next (Day 2):**
- Session 1: Structured Outputs -- JSON mode, function calling, Pydantic
- Session 2: LangChain -- LCEL chains, custom consulting tools
- Session 3: LangGraph -- StateGraph, conditional edges, cycles
- Session 4: Multi-Agent -- supervisor-worker teams